In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
bronze_stream_df = spark.readStream \
        .format("delta") \
        .table("airspace_pulse.bronze.flight_states")

In [0]:
silver_mapped_df = bronze_stream_df.select(
    col("time").cast(LongType()).cast(TimestampType()).alias("time"),
    col("flight_vector").getItem(0).alias("icao24"),
    trim(col("flight_vector").getItem(1)).alias("callsign"),
    col("flight_vector").getItem(2).alias("origin_country"),
    col("flight_vector").getItem(3).cast(LongType()).cast(TimestampType()).alias("time_position"),
    col("flight_vector").getItem(4).cast(LongType()).cast(TimestampType()).alias("last_contact"),
    col("flight_vector").getItem(5).cast(FloatType()).alias("longitude"),
    col("flight_vector").getItem(6).cast(FloatType()).alias("latitude"),
    col("flight_vector").getItem(7).cast(FloatType()).alias("baro_altitude"),
    col("flight_vector").getItem(8).cast(BooleanType()).alias("on_ground"),
    col("flight_vector").getItem(9).cast(FloatType()).alias("velocity"),
    col("flight_vector").getItem(10).cast(FloatType()).alias("true_track"),
    col("flight_vector").getItem(11).cast(FloatType()).alias("vertical_rate"),
    from_json(col("flight_vector").getItem(12), ArrayType(IntegerType())).alias("sensors"),
    col("flight_vector").getItem(13).cast(FloatType()).alias("geo_altitude"),
    col("flight_vector").getItem(14).alias("squawk"),
    col("flight_vector").getItem(15).cast(BooleanType()).alias("spi"),
    col("flight_vector").getItem(16).cast(IntegerType()).alias("position_source"),
    # Safe handling if category (index 17) is missing in some message batches
    get(col("flight_vector"), lit(17)).cast(IntegerType()).alias("category"),
    col("kafka_offset").cast(LongType()).alias("kafka_offset"),
    col("kafka_timestamp").cast(TimestampType()).alias("kafka_timestamp")
)

In [0]:
query = silver_mapped_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .option("checkpointLocation", "s3://airspace-lakehouse/airspace_pulse/_checkpoints/silver_flight_states") \
    .toTable("airspace_pulse.silver.flight_states_enriched")

query.awaitTermination()